# 학교 계정 · Dunnhumby GCN · seed50 · 8모형

완료된 seed49와 별도로 **seed50 하나만** M1/M2/M3/M4/M5_A/B/C/D를 새로 학습합니다. NGCF·GraphSAGE 실행과 출력 폴더를 분리합니다.

- 기존 GCN 모형 그대로: ID64·2층64, 자기연결1 추가 후 대칭 정규화, 은닉 ReLU/최종 선형64차원, dropout0, 층평균 없음.
- DAY1~704 학습 / DAY705~711 test, 신규상품 추천, MIN_ITEM_INTER=1. batch8192, Adam lr.0005, ID L2.001, 공유/GCN L2각1e-6, uniform 음성1개.
- 기존 M2 표현·M3 중심화N/V 그래프·M4 고객 내 평균1 가중치와 모든 조합 유지. 하나의 optimizer 공동학습. 입력 검증에는 기존 seed49 참조를 쓰며 성능값은 재사용하지 않습니다.
- 각300epoch 마지막점만 test평가. validation/holdout/조기종료/중간test 없음. 결과를 보고 설정을 바꾸지 않습니다.

## 10GB 학교 Drive 저장 구성
- 학교 Drive 최상위에 준비된 입력 ZIP은 약26.6MB입니다. 원본 바이트와 참조 파일을 SHA256으로 검증하고 **압축 해제는 Colab 임시 디스크에서만** 합니다. 초기화 후 같은 ZIP에서 다시 준비합니다.
- 진행 중 모형은 매epoch 전체 모델·Adam·난수를 저장합니다. 완료 후 결과를 기록한 모형만 Adam을 제외하고 정확한 최종 가중치·이력·난수를 보존합니다. 완료모형은 재학습·재평가하지 않습니다.
- 이번 실행의 체크포인트 예상 점유는 완료 약0.2GB, 교체 시 약0.35GB입니다. 입력 ZIP 포함 보수적으로0.75GB 예산을 둡니다. 실제 값은 저장 기록에 남깁니다.
- 시작 전과 매epoch 저장 전에 **학교 계정 실제 할당량과10GB 중 작은 값**을 기준으로 검사하고 추가1GB 여유를 남깁니다. 부족하면 이전 저장본을 보존하고 중단합니다. 다른 실행의 갑작스러운 쓰기·Drive 반영 지연까지 절대 보장하지는 않습니다.
- 기존 H&M 입력·결과를 삭제하지 않습니다. 전체142지표·비교·학습이력 ZIP에는 대용량 가중치를 중복 포함하지 않습니다.

학교 계정 `jungunb@u.sogang.ac.kr`의 **L4·고용량 RAM**으로 위에서부터 실행하세요. 끊기면 같은 링크로 다시 실행하며 마지막 저장 epoch부터 이어집니다. 동일 출력 폴더의 동시 실행은 금지합니다. 평가 직후 결과 저장 전에 중단되면 최종 평가가 재계산될 수 있습니다.

단일 추가시드이며 약10시드 최종평균·유의성 검증을 대신하지 않습니다. CPU 합성검사와 입력 원본 해시는 검증했으며 실제 학교 Colab seed50 전체 학습은 아직 실행하지 않았습니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 L4 GPU로 설정하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU: {GPU}. L4로 변경 후 실행하세요.'
SOURCE_COMMIT = '44171532fbff1e8ec550fb7e66ba62ab9eb713be'
REPO = Path('/content/clv-dh-gcn-s50-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 학교 계정·용량 확인
Drive 연결과 동일한 학교 계정으로 인증하세요.

In [ ]:
# Authenticate with the SAME account selected for drive.mount above.
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default()
drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)
OTHER_RUN_RESERVE_GB = 1.0  # Keep while GraphSAGE shares this Drive.

def check_space(required_bytes):
    info = drive_api.about().get(fields='user(emailAddress),storageQuota').execute()
    quota = info.get('storageQuota', {})
    if 'limit' not in quota or 'usage' not in quota:
        raise RuntimeError('계정 잔여 용량을 확인할 수 없습니다. 학습을 시작하지 않습니다.')
    assert info['user']['emailAddress'] == 'jungunb@u.sogang.ac.kr', '학교 계정으로 인증하세요.'
    available = min(int(quota['limit']), 10 * 10**9) - int(quota['usage'])
    needed = required_bytes + int(OTHER_RUN_RESERVE_GB * 10**9)
    if available < needed:
        raise RuntimeError(f"Drive 공간 부족 예방: {info['user']['emailAddress']} / "
                           f"남음 {available/1e9:.2f}GB, 필요한 여유 {needed/1e9:.2f}GB. "
                           '여유 있는 개인 Drive로 실행하거나 다른 실험 결과를 안전하게 보관한 뒤 재개하세요.')
    return info['user']['emailAddress'], available

account, available = check_space(750 * 10**6)
print('용량 확인 계정:', account, '| 잔여 GB:', round(available / 1e9, 2))
# Verify that the mounted MyDrive and quota API belong to the same account.
import uuid, time
probe = Path('/content/drive/MyDrive') / ('.clv_gcn_account_' + uuid.uuid4().hex)
probe.write_text('GCN account verification')
try:
    found = []
    for attempt in range(10):
        found = drive_api.files().list(q=f"name = '{probe.name}' and trashed = false and 'me' in owners",
                                      fields='files(id)', pageSize=10).execute().get('files', [])
        if found: break
        time.sleep(2)
    assert len(found) == 1, 'Drive 마운트와 용량 확인 계정이 다르거나 동기화가 지연되었습니다. 같은 계정으로 연결 후 재시도하세요.'
finally:
    probe.unlink(missing_ok=True)  # Only this newly created verification marker.


## 입력 검증·준비
기존 seed49 참조는 고정 입력 검증에만 사용합니다. 이번 학습·저장·평가는 seed50입니다.

In [ ]:
import hashlib, shutil, zipfile
import clv_dh_gcn_final8_seed50 as screen
assert Path(screen.__file__).resolve().parent == REPO.resolve(), '이 런타임을 재시작하고 위에서부터 실행하세요.'
BUNDLE = Path('/content/drive/MyDrive/dh_gcn_school_inputs_v1.zip')
EXPECTED = '22cbe0f0e6e0cb772e1391533e3fd6526e2fa42a0c5e2c5b4b2807c376b62828'
def sha(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(8*1024**2), b''):h.update(block)
    return h.hexdigest()
assert BUNDLE.is_file(), '학교 Drive 최상위의 준비된 dh_gcn_school_inputs_v1.zip을 확인하세요.'
assert sha(BUNDLE) == EXPECTED, '입력 ZIP 해시 불일치'
LOCAL = Path('/content/dh_gcn_school_inputs_v1')
LOCAL.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    manifest = json.loads(z.read('manifest.json'))
    assert set(manifest) == {'transaction_data.csv','product.csv','m2_reference.zip','m4_reference.zip'}
    for name, expected in manifest.items():
        target = LOCAL / name
        if not target.is_file() or sha(target) != expected:
            temporary = target.with_suffix('.partial')
            with z.open(name) as source, temporary.open('wb') as sink:shutil.copyfileobj(source,sink)
            assert sha(temporary) == expected, '원본 복사 불일치: ' + name
            temporary.replace(target)
screen.base.v3.SCHEMA['dunnhumby'].update(tx_path=str(LOCAL/'transaction_data.csv'),item_meta_path=str(LOCAL/'product.csv'))
cfg = screen.configure(out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_gcn_final8_seed50_school_v1',
                       m2_reference=str(LOCAL/'m2_reference.zip'),m4_reference=str(LOCAL/'m4_reference.zip'))
def storage_guard(required_bytes):
    check_space(required_bytes)
    root = Path(cfg.out_dir)
    used = sum(p.stat().st_size for p in root.rglob('*') if p.is_file()) if root.exists() else 0
    if used + required_bytes + BUNDLE.stat().st_size > 750 * 10**6:
        raise RuntimeError('이번 GCN 저장 예산0.75GB 초과 예방: 기존 파일을 보존하고 중단합니다.')
screen.STORAGE_GUARD = storage_guard
storage_guard(150 * 10**6)
prepared = screen.prepare(cfg)
print('원본 검증 완료 / seed50 / 8모형 / 각300epoch:', prepared['run_dir'])


## 2. GCN 8개 모형 순차 학습
모형당 시작3개 epoch 및 이후5epoch마다 손실·전체 학습 시간·Drive 저장 시간을 표시합니다. 처음5배치가 끝나면 입력/음성추출과 학습계산의 배치당 표본 시간도 먼저 표시하므로 첫epoch가 끝나기 전에도 확인할 수 있습니다.

매epoch checkpoint를 Drive에 저장하고, 각 모형300epoch 완료 시 전체142지표를 평가·저장한 뒤 다음 모형으로 넘어갑니다. **로그 빈도와 무관하게 Drive 저장은 매epoch 실행**됩니다. 학습 중 다른 셀을 실행하거나 런타임을 바꾸지 마세요.

각 시간은 다음과 같습니다.
- 입력·음성추출: 같은 순서의 uniform 음성표본 생성과 배치 자료 준비/전송.
- 학습계산: 그래프 전파·BPR 손실·역전파·Adam 업데이트.
- 학습 전체: epoch의 입력·계산·로그/진행기록을 포함한 벽시계 시간.
- Drive 저장완료: 로컬 직렬화·복사/해시·Drive 저장/메타데이터 전체 시간.


In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

## 시간 기록 확인 — 학습·평가 없음
아래 셀은 저장된 시간 CSV만 읽습니다. 학습 셀이 끝났거나 중단된 뒤 실행하세요. 마지막10개 epoch를 보여주며 전체 기록은 결과 ZIP의 `timing/`에 포함됩니다. 입력 준비 총 시간은 `preparation_timing.json`에 있습니다. 표본 시간은 첫3개 epoch만 있으며, 나머지 빈칸은 미측정입니다.

In [ ]:
import pandas as pd
from IPython.display import display
timing_files = sorted((prepared['run_dir'] / 'progress' / 'timing').glob('*.csv'))
if timing_files:
    timing = pd.concat([pd.read_csv(p) for p in timing_files], ignore_index=True)
    columns = ['model_id', 'epoch', 'train_seconds', 'save_seconds',
               'sample_input_seconds_per_batch', 'sample_compute_seconds_per_batch',
               'checkpoint_local_seconds', 'checkpoint_drive_seconds', 'checkpoint_metadata_seconds']
    display(timing[columns].tail(10))
    print('단위: 초. 표본 계측은 추가 동기화 비용을 포함합니다. test 성능을 측정하지 않습니다.')
else:
    print('아직 완료 epoch의 시간 기록이 없습니다. 초기 배치 시간은 학습 출력에서 확인하세요.')

## 3. 저장된 완료 모형만 비교 — 학습·재평가 없음
8모형 절대지표와 **같은 GCN M1 대비** 변화율입니다. 미완료 모형은 빈칸이며0점이 아닙니다. 전체142지표·모든 모형 간 비교·개입 진단은 CSV에 보관됩니다. 단일 시드이므로 표준편차/신뢰구간을 만들지 않습니다.

세션이 끊겼다면1번 준비 셀까지 실행한 다음 이 셀로 바로 와도 됩니다. 전체 결과를 받으려면2번 학습 셀을 이어서 실행하세요.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = list(screen.CORE)
display(result['absolute'].set_index('model')[metrics].T.reindex(columns=screen.ORDER))
against_m1 = result['comparison'].query("reference == 'M1'")
if not against_m1.empty:
    display(against_m1.pivot(index='metric', columns='model', values='relative_change_pct')
            .reindex(index=metrics, columns=screen.ORDER[1:]))
    print('GCN M1 대비 상대변화(%). 모든 비교·불리한 지표도 CSV에 보존합니다.')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
print('seed50 1시드 — 평균/표준편차·유의성·CLV 배정 귀속 판단 아님')

## 4. 결과 ZIP 다운로드
완료 모형의 원본 결과·1~300epoch 학습 이력·전체 비교·설정·입력 검증 기록을 받습니다. 8개 미완료 시 파일명에 `_partial`을 붙입니다. 재개용 대용량 checkpoint는 ZIP에 넣지 않고 **Drive에 그대로 유지**합니다.

In [ ]:
from google.colab import files
import shutil
output = screen.export_zip(prepared, '/content/dh_gcn_final8_seed50_results.zip')
src = Path(output)
backup = prepared['run_dir'] / src.name
shutil.copy2(src, backup)
assert backup.stat().st_size == src.stat().st_size
print('Drive 보관:', backup, '| ZIP 크기(MB):', round(src.stat().st_size / 1024**2, 3))
print('다운로드가 멈추면 위 Drive 경로에서 직접 다운로드하세요. checkpoint는 Drive에 유지됩니다.')
files.download(str(src))